In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [62]:
df = pd.read_csv('heart.dat', sep = ' ', header = None)

In [63]:
train = df.loc[0:int(0.2*len(df)),]
test = df.loc[int(0.2*len(df))+1:,]

In [64]:
X_train = pd.DataFrame()
X_test = pd.DataFrame()
for i in range(13):
    X_train[i] = train[i]
    X_test[i] = test[i]

y_train = pd.DataFrame(data=train[13])
y_test = pd.DataFrame(data=test[13])

In [65]:
y_train = y_train.squeeze()
classes = y_train.unique()
n = X_train.shape[1]

priors = {c: (y_train == c).mean() for c in classes}

params = {}
for c in classes:
    X_c = X_train[y_train == c]
    mu_c = X_c.mean().values
    sigma_c = np.cov(X_c.T)
    params[c] = (mu_c, sigma_c)

def pdf(x, mu, sigma, n):
    const = 1 / np.sqrt(((2*np.pi)**n) * np.linalg.det(sigma))
    diff = (x - mu).reshape(-1, 1)
    expo = -0.5 * diff.T @ np.linalg.inv(sigma) @ diff
    return float(const * np.exp(expo))

y_pred = []

for i, row in X_test.iterrows():
    x = row.values
    posteriors = {}
    for c in classes:
        mu, sigma = params[c]
        likelihood = pdf(x, mu, sigma, n)
        posteriors[c] = likelihood * priors[c]
        
    y_pred.append(max(posteriors, key=posteriors.get))

y_pred = pd.Series(y_pred, index=X_test.index)

/tmp/ipykernel_1930/3711126876.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(const * np.exp(expo))


In [66]:
from sklearn.metrics import accuracy_score

print("Acurácia:", accuracy_score(y_test, y_pred))
print("Matriz de confusão:\n", confusion_matrix(y_test, y_pred))

Acurácia: 0.7674418604651163
Matriz de confusão:
 [[94 25]
 [25 71]]
